# BirdCLEF 2026 — Two-Stage SED + Species Classifier — Inference Only (Pipeline 05)

This inference notebook loads two models:
1. **SED CRNN**: Filters out inactive segments (saving compute and reducing false positives).
2. **Species Classifier**: Predicts specific birds only on active segments.

In [ ]:
import os, gc, math, glob, numpy as np, pandas as pd, soundfile as sf
from tqdm.auto import tqdm
import torch, torch.nn as nn, torch.nn.functional as F
import torchaudio.transforms as T
import timm
import warnings; warnings.filterwarnings('ignore')

class Config:
    ROOT_DIR = '/kaggle/input/competitions/birdclef-2026'
    TRAIN_CSV = os.path.join(ROOT_DIR, 'train.csv')
    SOUNDSCAPE_DIR = os.path.join(ROOT_DIR, 'train_soundscapes')
    SED_MODEL_PATH = 'best_sed_crnn.pth'
    SPEC_MODEL_PATH = 'best_species_classifier.pth'
    SR = 32000
    WINDOW_SECONDS = 5
    N_MELS, N_FFT, HOP_LENGTH, FMIN, FMAX = 128, 2048, 512, 20, 16000
    MODEL_NAME = 'tf_efficientnet_b0'
    SED_THRESHOLD = 0.5  # If prob < threshold, predict all zeros

CFG = Config()

sample_sub = pd.read_csv(os.path.join(CFG.ROOT_DIR, 'sample_submission.csv'))
submission_labels = [c for c in sample_sub.columns if c != 'row_id']
CFG.NUM_CLASSES = len(submission_labels)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


In [ ]:
class SED_CRNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1), nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.rnn = nn.GRU(input_size=128 * (CFG.N_MELS // 16), hidden_size=128, bidirectional=True, batch_first=True)
        self.fc = nn.Linear(256, 1)
        
    def forward(self, x):
        x = self.cnn(x)
        B, C, F, T = x.shape
        x = x.permute(0, 3, 1, 2).contiguous().view(B, T, C * F)
        x, _ = self.rnn(x)
        x, _ = torch.max(x, dim=1)
        return self.fc(x)

class SpeciesClassifier(nn.Module):
    def __init__(self, model_name, num_classes):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=False, in_chans=3)
        if 'efficientnet' in model_name:
            in_features = self.backbone.classifier.in_features
            self.backbone.classifier = nn.Identity()
        else:
            in_features = self.backbone.get_classifier().in_features
            self.backbone.reset_classifier(0)
        self.head = nn.Linear(in_features, num_classes)
    def forward(self, x): return self.head(self.backbone(x))

sed_model = SED_CRNN().to(device)
spec_model = SpeciesClassifier(CFG.MODEL_NAME, CFG.NUM_CLASSES).to(device)

try:
    sed_model.load_state_dict(torch.load(CFG.SED_MODEL_PATH, map_location=device))
    spec_model.load_state_dict(torch.load(CFG.SPEC_MODEL_PATH, map_location=device))
    print('Both models loaded successfully.')
except:
    print('Fallback: Models missing, using random weights.')

sed_model.eval()
spec_model.eval()


In [ ]:
TEST_DIR = os.path.join(CFG.ROOT_DIR, 'test_soundscapes')
test_files = sorted(glob.glob(f'{TEST_DIR}/*.ogg')) if os.path.exists(TEST_DIR) else []
if not test_files:
    print('FALLBACK: Using train soundscapes')
    test_files = sorted(glob.glob(f'{CFG.SOUNDSCAPE_DIR}/*.ogg'))[:5]

mel_transform = T.MelSpectrogram(sample_rate=CFG.SR, n_fft=CFG.N_FFT, hop_length=CFG.HOP_LENGTH, n_mels=CFG.N_MELS, f_min=CFG.FMIN, f_max=CFG.FMAX).to(device)
amplitude_to_db = T.AmplitudeToDB(top_db=80).to(device)

all_preds, all_row_ids = [], []
skipped_count = 0

for audio_path in tqdm(test_files):
    filename = os.path.basename(audio_path).replace('.ogg', '')
    try: y, _ = sf.read(audio_path, always_2d=True); y = y.mean(axis=1)
    except: continue
    y_t = torch.tensor(y, dtype=torch.float32).to(device)
    window_samples = CFG.SR * CFG.WINDOW_SECONDS
    for seg_idx in range(math.ceil(len(y_t) / window_samples)):
        start_sample = seg_idx * window_samples
        segment = y_t[start_sample : start_sample + window_samples]
        if len(segment) < window_samples: segment = F.pad(segment, (0, window_samples - len(segment)))
        
        row_id = f'{filename}_{(seg_idx + 1) * CFG.WINDOW_SECONDS}'
        all_row_ids.append(row_id)

        with torch.no_grad():
            mel = amplitude_to_db(mel_transform(segment))
            mel = (mel - mel.min()) / (mel.max() - mel.min() + 1e-6)
            img = torch.stack([mel, mel, mel]).unsqueeze(0)
            
            # Phase 1: Check if segment is active
            sed_prob = torch.sigmoid(sed_model(img)).item()
            if sed_prob < CFG.SED_THRESHOLD:
                # Save compute, predict all zeros
                all_preds.append(np.zeros(CFG.NUM_CLASSES))
                skipped_count += 1
            else:
                # Phase 2: Run full species classifier
                probs = torch.sigmoid(spec_model(img)).squeeze(0).cpu().numpy()
                all_preds.append(probs)

print(f'Done! Skipped {skipped_count} inactive segments out of {len(all_row_ids)}.')

sub_df = pd.DataFrame(all_preds, columns=submission_labels)
sub_df.insert(0, 'row_id', all_row_ids)
sub_df.to_csv('submission.csv', index=False)
print('Submission saved!')
